In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json
/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json


In [2]:
# Install only if needed
!pip install -q --upgrade transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 91.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 36.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

import evaluate

from sklearn.model_selection import train_test_split

In [13]:
import pandas as pd

df = pd.read_json(
    "/kaggle/input/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json",
    lines=True
)

df.dropna(inplace=True)

print(df.shape)
df.head()


(28619, 3)


,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [19]:
print(df.columns)

Index(['is_sarcastic', 'headline', 'article_link'], dtype='object')


In [22]:
df = df.rename(columns={
    "headline": "text",
    "is_sarcastic": "label"
})

df = df[["text", "label"]]

In [23]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(train_df.shape)
print(val_df.shape)

(22895, 2)
(5724, 2)


In [24]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [26]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [27]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [28]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/22895 [00:00<?, ? examples/s]

Map:   0%|          | 0/5724 [00:00<?, ? examples/s]

In [29]:
train_dataset = train_dataset.remove_columns(["text"])

if "__index_level_0__" in train_dataset.column_names:
    train_dataset = train_dataset.remove_columns(["__index_level_0__"])

val_dataset = val_dataset.remove_columns(["text"])

if "__index_level_0__" in val_dataset.column_names:
    val_dataset = val_dataset.remove_columns(["__index_level_0__"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [30]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch",

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_steps=500,      # Replace warmup_ratio with warmup_steps

    num_train_epochs=5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    save_total_limit=1,

    report_to="none"
)

In [36]:
from transformers import Trainer, DataCollatorWithPadding, EarlyStoppingCallback

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [ ]:
trainer.train()

In [16]:
model.save_pretrained("/kaggle/working/bert_model")

tokenizer.save_pretrained("/kaggle/working/bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/bert_model/tokenizer_config.json',
 '/kaggle/working/bert_model/tokenizer.json')

In [18]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

trained_model = trainer.model
trained_model.to(device)
trained_model.eval()

text = "Wow! Another Monday morning. I am so excited."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = trained_model(**inputs)

prediction = outputs.logits.argmax(dim=1).item()

print("Prediction:", prediction)
print("Sarcastic" if prediction == 1 else "Not Sarcastic")

Prediction: 0
Not Sarcastic


In [19]:
trainer.save_model("/kaggle/working/fine_tuned_bert")

tokenizer.save_pretrained("/kaggle/working/fine_tuned_bert")

print("Fine-tuned model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully!


In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

fine_tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/working/fine_tuned_bert"
)

fine_model = AutoModelForSequenceClassification.from_pretrained(
    "/kaggle/working/fine_tuned_bert"
)

print("Fine-tuned model loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!


In [22]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fine_model.to(device)
fine_model.eval()

text = "I absolutely enjoy getting stuck in traffic every day."

inputs = fine_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = fine_model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

if prediction == 1:
    print("Sarcastic")
else:
    print("Not Sarcastic")

Not Sarcastic
